# Make 24 Demo
A class demonstration about combining digits to make the number 24. The goal here is to (1) demonstrate how a symbolic system can solve problems, (2) get hands-on experience making your own such system, and (3) observe the kind of result that led early cognitive scientists to think that intelligent systems were comprised of large amounts of symbolic knowledge. 

# Part I: Naive search
In this first section, you will write your own code to naively search all combinations of digits and operators until you find a matching pair, or exhaust all possibilities. Below is useful code that you can modify and reuse to write such a model.

In [ ]:
# if you are missing any of these libraries, you can install with...
# pip install [name of package]
import io
from contextlib import redirect_stdout
import matplotlib.pyplot as plt
import pandas as pd
from itertools import permutations, product


Using `permutations` as a shortcut for iterating through many permutations (as the name suggests):

In [ ]:
# RUN THIS CELL TO SEE HOW PERMUSTATIONS WORKS
orderings = permutations([1,2,3])

# note that permutations returns an iterator object, not an array
print('You can move through an iterator object with a loop:')
for order in orderings: # so you will need to call it in some way to get the contents
    print(order)

print('Note how it does not reuse orderings.')

The order of operations does matter, so for your convenience, here is a list of all the possible orderings. You might find this variable useful in writing your `make_24` function below. You are encourages to run this cell and use the variable `shapes`, but you are free to use another method if you like:

In [22]:
shapes = [
    "(({a} {x} {b}) {y} {c}) {z} {d}",   # ((a b) c) d
    "({a} {x} ({b} {y} {c})) {z} {d}",   # (a (b c)) d
    "({a} {x} {b}) {y} ({c} {z} {d})",   # (a b) (c d)
    "{a} {x} (({b} {y} {c}) {z} {d})",   # a ((b c) d)
    "{a} {x} ({b} {y} ({c} {z} {d}))",   # a (b (c d))
]

Below is some example code for three numbers. You can use this code as a reference for programming naive search for four numbers. Hint, you'll want to use `shapes` instead of `mini_shapes`, change `repeat`, and modify what the `for` loops iterate over. 

In [ ]:
# here is some example code that lists and evaluates three numbers
three_nums = [2,3,5]

# different orders of operations for three numberd
mini_shapes = [
    "({a} {x} {b}) {y} {c}",   # (a b) c
    "{a} {x} ({b} {y} {c})",   # a (b c)
]

def make_24_mini(three_nums, target=24):
    tried = 0
    for a, b, c in permutations(three_nums):
        for x, y in product("+-*/", repeat=2):
            for shape in mini_shapes:
                expression = shape.format(a=a, b=b, c=c, x=x, y=y)
                tried += 1
                try:
                    if abs(eval(expression) - 24) < 1e-9:
                        print(f"tried {tried}: found {expression}")
                        return expression, tried
                except ZeroDivisionError:
                    pass
    print(f"tried {tried}: no solution")
    return None, tried

### Make your own function
Make your own make_24 model. If you like, you can modify the above `mini_make_24` above to take 4 numbers instead of just three. The real goal here is to give you the chance to work through the logic yourself so that you can get a feel for how a model like this works (while still keeping this demo short).

In [ ]:
# SOLUTION - MAKE SURE YOU ARE IN THE WRITE NOTEBOOK IF YOU HAVE NOT ATTEMPTED ON YOUR OWN YET
def make_24(numbers, target=24):
    tried = 0
    for a, b, c, d in permutations(numbers):
        for x, y, z in product("+-*/", repeat=3):
            for shape in shapes:
                expression = shape.format(a=a, b=b, c=c, d=d, x=x, y=y, z=z)
                tried += 1
                try:
                    if abs(eval(expression) - 24) < 1e-9:
                        print(f"tried {tried}: found {expression}")
                        return expression, tried
                except ZeroDivisionError:
                    pass
    print(f"tried {tried}: no solution")
    return None, tried

### Try your code on a few puzzles
Exectue the following cell and see how your function does on a few puzzles.

In [ ]:
import re
from contextlib import redirect_stdout
import io
import pandas as pd

puzzles = [[1, 2, 3, 4], [4, 7, 8, 8], [2, 3, 4, 5], [5, 5, 5, 1], [3, 3, 8, 8], [1, 1, 1, 1]]

rows = []
for puzzle in puzzles:
    with redirect_stdout(io.StringIO()):   # hide make_24's own print lines
        result, tries = make_24(puzzle)

    if result is None:
        correct = "?"
    else:
        uses_puzzle = sorted(int(n) for n in re.findall(r"\d+", result)) == sorted(puzzle)
        correct = "yes" if uses_puzzle and abs(eval(result) - 24) < 1e-9 else "NO"

    rows.append({
        "Puzzle": " ".join(str(n) for n in puzzle),
        "Result": result if result is not None else "no solution",
        "Tries": tries,
        "Checks out": correct,
    })

pd.DataFrame(rows)

# Part II: Think on your own
The goal above was to give you a feel for how a naive search for solutions to "Make 24" might work. However, it seems unlikely that people solve the task this way. 
(1) Write a few sentences about how people might solve Make 24 differently. 
(2) In plain language (no need to write code), what is one change you might make to the model to make it more human like? (or one way to improve the model)
(3) Find a neighbor (or preceptor) that has finished this part, and compare your thoughts. 
(4) Once you have had a chat, move on to Part III.

SOLUTION REFERENCE

(1) 
* It seems like people might set subgoals to find certain number combinations
* These subgoals rely on knowledge about the limited ways to make 24
* People do not randomly try all combinations, and instead try promising avenues

(2) 
Adding subgoal search informed by knowledge about valid ways to make 24 would speed up the search

(3)
Enjoy conversation with a peer!

(4)
Move on to part iii

# Part III: A smarter model
Try not to read below before having a discussion about part II with someone else.

Now that we have discussed some idea about how to make this system smarter (and more human like), let's try one method. Here, we can add "knowledge" to the system like people might have learned in elementary school. 

Below, try adding some facts about how to make 24 from two numbers and one operator. Then run the below code to see if this more advanced model can make 24 in less attempts. 

In [ ]:
# Add more knowledge about how to make 24 here
known_24s = [
    (3, "*", 8), 
    (4, "*", 6), 
    (2, "*", 12), 
    (12, "+", 12), 
    (20, "+", 4), 
    (30, "-", 6)
    ]

In [ ]:
def make_target(three, target):
    """The three-number example from above, made into a function with any target."""
    tried = 0
    for a, b, c in permutations(three):
        for x, y in product("+-*/", repeat=2):
            for shape in mini_shapes:
                expression = shape.format(a=a, b=b, c=c, x=x, y=y)
                tried += 1
                try:
                    if abs(eval(expression) - target) < 1e-9:
                        return expression, tried
                except ZeroDivisionError:
                    pass
    return None, tried

def make_24_smart(numbers):
    tried = 0
    for left, op, right in known_24s: # use knowledge to create targets
        for keep, need in ((left, right), (right, left)):
            if keep in numbers:
                rest = list(numbers)
                rest.remove(keep)
                print(f"I have a {keep}. Can {rest} make {need}?")
                sub, t = make_target(rest, need)
                tried += t
                if sub is not None:
                    print(f"  yes: {sub} = {need}")
                    expression = f"{keep} {op} ({sub})" if keep == left else f"({sub}) {op} {keep}"
                    return expression, tried
                print("  no")
    print("No known fact worked. Falling back to the full search.")
    result, t = make_24(numbers)
    return result, tried + t

In [44]:
make_24_smart([4, 7, 8, 8])

I have a 8. Can [4, 7, 8] make 3?
  yes: (4 + 7) - 8 = 3


('((4 + 7) - 8) * 8', 3)

Run the cell below to see how the smart and naive model hold up. Notice that for many the model might seriously improve at some problems, while some are still very hard for the model to solve. 

In [ ]:
puzzles = [[4, 7, 8, 8], [2, 2, 6, 6], [4, 6, 1, 1], [3, 8, 2, 2], [6, 6, 2, 1],
           [1, 2, 3, 4], [9, 5, 3, 1], [10, 10, 4, 4], [3, 3, 8, 8], [1, 1, 1, 1]]

rows = []
for puzzle in puzzles:
    with redirect_stdout(io.StringIO()):          # hide the printed reasoning
        naive_result, naive_tries = make_24(puzzle)
        smart_result, smart_tries = make_24_smart(puzzle)
    rows.append({
        "Puzzle": " ".join(str(n) for n in puzzle),
        "make_24": naive_tries,
        "make_24_smart": smart_tries,
    })
table = pd.DataFrame(rows).set_index("Puzzle")

wins = (table["make_24_smart"] < table["make_24"]).sum()
ax = table.plot.bar(figsize=(10, 4), logy=True, rot=0)
ax.set_ylabel("Tries (log scale)")
ax.set_title(f"Make 24 Attempts with and Without Fact Recall"
             f"( {len(known_24s)} known facts)")
plt.show()
table

### Challenging problems
Can you get the model to improve at [3,3,8,8] by adding knowledge? If yes, what knowledge did you add? If not, what might this say about why people have trouble solving this problem?

SOLUTION REFERENCE
Getting the problem to solve this problem well might require knowledge about non-integers. This might not be that big of a problem, but this kind of knowledge might be less available to people, which might explain why this particular problem is harder for people and the agent alike. 